# SBC Validation: 2D Correlated Gaussian

Simulation-Based Calibration (SBC) for PTSampler, NUTSSampler, and RJPTSampler (MH-only)
on a 2D correlated Gaussian problem with known analytic posterior.

**Prior:** $\theta \sim \mathcal{N}(0, \Sigma_{\text{prior}})$, $\Sigma_{\text{prior}} = \begin{pmatrix} 4 & 1 \\ 1 & 4 \end{pmatrix}$

**Likelihood:** $d \mid \theta \sim \mathcal{N}(\theta, \Sigma_{\text{noise}})$, $\Sigma_{\text{noise}} = \begin{pmatrix} 1 & 0.5 \\ 0.5 & 1 \end{pmatrix}$

**Posterior:** $\mathcal{N}(\mu_{\text{post}}, \Sigma_{\text{post}})$ where $\Sigma_{\text{post}}^{-1} = \Sigma_{\text{prior}}^{-1} + \Sigma_{\text{noise}}^{-1}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from impulse import (
    PTSampler,
    NUTSSampler,
    RJPTSampler,
    compose_logp_and_grad,
    run_sbc_continuous,
    sbc_ecdf_plot,
    coverage_plot,
    rank_histogram,
)

plt.rcParams.update({"figure.dpi": 120})

## 1. Define Model

In [ ]:
NDIM = 2
PARAM_NAMES = [r"$\theta_0$", r"$\theta_1$"]

# Covariance matrices
Sigma_prior = np.array([[4.0, 1.0], [1.0, 4.0]])
Sigma_noise = np.array([[1.0, 0.5], [0.5, 1.0]])

# Precompute inverses for likelihood / prior
Sigma_prior_inv = np.linalg.inv(Sigma_prior)
Sigma_noise_inv = np.linalg.inv(Sigma_noise)
log_norm_prior = -0.5 * (NDIM * np.log(2 * np.pi) + np.linalg.slogdet(Sigma_prior)[1])
log_norm_noise = -0.5 * (NDIM * np.log(2 * np.pi) + np.linalg.slogdet(Sigma_noise)[1])


class GaussianPrior:
    """log N(0, Sigma_prior)."""
    def __call__(self, x):
        return log_norm_prior - 0.5 * x @ Sigma_prior_inv @ x


class GaussianPriorGrad:
    """Gradient of log N(0, Sigma_prior)."""
    def __call__(self, x):
        return -Sigma_prior_inv @ x


class GaussianLikelihood:
    """log N(theta, Sigma_noise) evaluated at a single data point."""
    def __init__(self, data):
        self.data = data

    def __call__(self, x):
        r = self.data - x
        return log_norm_noise - 0.5 * r @ Sigma_noise_inv @ r


class GaussianLikelihoodGrad:
    """Gradient of the Gaussian likelihood w.r.t. theta."""
    def __init__(self, data):
        self.data = data

    def __call__(self, x):
        r = self.data - x
        return Sigma_noise_inv @ r


def prior_draw(rng):
    """Draw from the prior."""
    return rng.multivariate_normal(np.zeros(NDIM), Sigma_prior)


def data_generator(theta_true, rng):
    """Generate a single data point from the likelihood."""
    return rng.multivariate_normal(theta_true, Sigma_noise)

## 2. Sampler Factories

In [ ]:
def pt_factory(data, rng, outdir):
    """Create and run a PTSampler."""
    lnlike = GaussianLikelihood(data)
    lnprior = GaussianPrior()
    sampler = PTSampler(
        ndim=NDIM,
        lnlike=lnlike,
        lnprior=lnprior,
        ntemps=5,
        seed=int(rng.integers(0, 2**31)),
        outdir=outdir,
        save_freq=2000,
    )
    x0 = prior_draw(rng)
    sampler.sample(x0, num_iterations=2000)
    return sampler


def nuts_factory(data, rng, outdir):
    """Create and run a NUTSSampler."""
    lnlike = GaussianLikelihood(data)
    lnprior = GaussianPrior()
    lnlike_grad = GaussianLikelihoodGrad(data)
    lnprior_grad = GaussianPriorGrad()
    logp_and_grad = compose_logp_and_grad(
        lnlike, lnprior, lnlike_grad=lnlike_grad, lnprior_grad=lnprior_grad
    )
    sampler = NUTSSampler(
        ndim=NDIM,
        logp_and_grad=logp_and_grad,
        num_warmup=500,
        seed=int(rng.integers(0, 2**31)),
        outdir=outdir,
        save_freq=1000,
    )
    x0 = prior_draw(rng)
    sampler.sample(x0, num_iterations=1000)
    return sampler


def rjpt_factory(data, rng, outdir):
    """Create and run an RJPTSampler (MH-only, no gradient)."""
    lnlike = GaussianLikelihood(data)
    lnprior = GaussianPrior()
    sampler = RJPTSampler(
        ndim=NDIM,
        lnlike=lnlike,
        lnprior=lnprior,
        ntemps=5,
        seed=int(rng.integers(0, 2**31)),
        outdir=outdir,
        save_freq=2000,
    )
    x0 = prior_draw(rng)
    sampler.sample(x0, num_iterations=2000)
    return sampler

## 3. Run SBC (200 simulations each)

In [ ]:
N_SIM = 200
SEED = 42

In [ ]:
print("=== PTSampler ===")
pt_results = run_sbc_continuous(
    sampler_factory=pt_factory,
    prior_draw=prior_draw,
    data_generator=data_generator,
    n_simulations=N_SIM,
    burn=500,
    seed=SEED,
)

In [ ]:
print("=== NUTSSampler ===")
nuts_results = run_sbc_continuous(
    sampler_factory=nuts_factory,
    prior_draw=prior_draw,
    data_generator=data_generator,
    n_simulations=N_SIM,
    burn=200,
    seed=SEED + 1,
)

In [ ]:
print("=== RJPTSampler (MH) ===")
rjpt_results = run_sbc_continuous(
    sampler_factory=rjpt_factory,
    prior_draw=prior_draw,
    data_generator=data_generator,
    n_simulations=N_SIM,
    burn=500,
    seed=SEED + 2,
)

## 4. Figure 1: ECDF Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, results, title in zip(
    axes,
    [pt_results, nuts_results, rjpt_results],
    ["PTSampler", "NUTSSampler", "RJPTSampler (MH)"],
):
    sbc_ecdf_plot(results["quantiles"], param_names=PARAM_NAMES, ax=ax)
    ax.set_title(title)

fig.suptitle("SBC ECDF — 2D Correlated Gaussian", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

## 5. Figure 2: Coverage Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, results, title in zip(
    axes,
    [pt_results, nuts_results, rjpt_results],
    ["PTSampler", "NUTSSampler", "RJPTSampler (MH)"],
):
    coverage_plot(results["quantiles"], ax=ax)
    ax.set_title(title)

fig.suptitle("Coverage — 2D Correlated Gaussian", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

## 6. Figure 3: Rank Histograms

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 10))

for row, results, title in zip(
    axes,
    [pt_results, nuts_results, rjpt_results],
    ["PTSampler", "NUTSSampler", "RJPTSampler (MH)"],
):
    rank_histogram(
        results["ranks"],
        n_posterior_samples=results["n_posterior"],
        param_names=PARAM_NAMES,
        axes=row,
    )
    row[0].set_ylabel(title)

fig.suptitle("Rank Histograms — 2D Correlated Gaussian", y=1.01, fontsize=14)
fig.tight_layout()
plt.show()

## 7. Summary: KS Test

In [ ]:
print(f"{'Sampler':<22} {'Param':<10} {'KS stat':>10} {'p-value':>10}")
print("-" * 55)

for name, results in [
    ("PTSampler", pt_results),
    ("NUTSSampler", nuts_results),
    ("RJPTSampler (MH)", rjpt_results),
]:
    for j, pname in enumerate(PARAM_NAMES):
        stat, pval = stats.kstest(results["quantiles"][:, j], "uniform")
        flag = "" if pval > 0.05 else " ***"
        print(f"{name:<22} {pname:<10} {stat:>10.4f} {pval:>10.4f}{flag}")